# 03 — Reinforcement Learning Environment & Agent Development
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Phase:** Reinforcement Learning Formulation, Environment Design & Agent Architecture

---

## Academic Formulation: MDP for Algorithmic Trading

In this framework, trading is formulated as a Markov Decision Process (MDP) defined by the tuple $\langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle$:

1. **State Space $\mathcal{S}$:**
   The composite state $s_t = [e_t \,;\, p_t]$ consists of:
   - $e_t \in \mathbb{R}^{32}$: Latent market state embedding from the pre-trained `MarketStateLSTMEncoder`
   - $p_t \in \mathbb{R}^{K}$: Internal portfolio state (current position weight, cash fraction, unrealized PnL, holding duration)

2. **Action Space $\mathcal{A}$:**
   - Discrete: $\mathcal{A} \in \{-1, 0, 1\}$ (Short, Flat, Long) or $\{0, 1, 2\}$
   - Continuous: $a_t \in [-1.0, 1.0]$ target portfolio weight

3. **Reward Function $\mathcal{R}(s_t, a_t, s_{t+1})$:**
   - Differential Sharpe Ratio or log utility:
   $$R_t = a_t \cdot r_{t+1} - c \cdot |a_t - a_{t-1}|$$
   where $r_{t+1}$ is the forward return and $c$ is the transaction cost fee factor (e.g., 10 bps).

4. **Discount Factor $\gamma$:** $\gamma \in [0.95, 0.99]$

## 1. Imports & Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

CHECKPOINT_PATH = os.path.join("backend", "checkpoints", "best_rnn_encoder.pt")
EMBEDDINGS_DIR  = os.path.join("backend", "embeddings")
TRAIN_DIR       = os.path.join("data", "training", "train")
VAL_DIR         = os.path.join("data", "training", "validation")
TEST_DIR        = os.path.join("data", "training", "test")

print(f"Checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}")
print(f"Embeddings dir exists: {os.path.exists(EMBEDDINGS_DIR)}")

## 2. Inspect Pre-computed Market State Embeddings

In [ ]:
# Verify embeddings generated from Phase 4/5
train_emb = torch.load(os.path.join(EMBEDDINGS_DIR, "train_embeddings.pt"), weights_only=True)
val_emb   = torch.load(os.path.join(EMBEDDINGS_DIR, "val_embeddings.pt"), weights_only=True)
test_emb  = torch.load(os.path.join(EMBEDDINGS_DIR, "test_embeddings.pt"), weights_only=True)

print("Pre-computed Market State Embeddings:")
print(f"  Train: shape={tuple(train_emb.shape)}, dtype={train_emb.dtype}")
print(f"  Val  : shape={tuple(val_emb.shape)}, dtype={val_emb.dtype}")
print(f"  Test : shape={tuple(test_emb.shape)}, dtype={test_emb.dtype}")
assert train_emb.shape[1] == 32, "Expected 32-D embedding dimension"
print("Market state embeddings dimensionality verified: 32 dimensions ✓")

## 3. Trading Environment Interface (Scaffolding)

In [ ]:
class MarketTradingEnv:
    """
    Algorithmic Trading Gym-compatible Environment.
    
    Combines:
      - 32-D RNN Market State Embeddings
      - Target Forward Returns
      - Portfolio Account State (cash, position, portfolio value)
      - Realistic Transaction Costs & Slippage
    """
    def __init__(
        self,
        embeddings: np.ndarray,
        returns: np.ndarray,
        initial_capital: float = 100000.0,
        transaction_cost_pct: float = 0.001,  # 10 bps
    ):
        self.embeddings = embeddings
        self.returns = returns
        self.initial_capital = initial_capital
        self.transaction_cost_pct = transaction_cost_pct
        self.n_steps = len(returns)
        self.current_step = 0
        self.position = 0.0  # -1.0 to +1.0
        self.cash = initial_capital
        self.portfolio_value = initial_capital
        
    def reset(self):
        self.current_step = 0
        self.position = 0.0
        self.cash = self.initial_capital
        self.portfolio_value = self.initial_capital
        return self._get_observation()
        
    def _get_observation(self):
        market_state = self.embeddings[self.current_step]
        portfolio_state = np.array([self.position, self.portfolio_value / self.initial_capital], dtype=np.float32)
        return np.concatenate([market_state, portfolio_state])
        
    def step(self, action: float):
        # Clip action to [-1, 1]
        action = float(np.clip(action, -1.0, 1.0))
        delta_pos = abs(action - self.position)
        cost = delta_pos * self.portfolio_value * self.transaction_cost_pct
        
        # Step forward return
        ret = self.returns[self.current_step]
        pnl = self.position * self.portfolio_value * ret - cost
        self.portfolio_value += pnl
        self.position = action
        
        # Immediate reward (transaction-cost adjusted return)
        reward = (pnl / self.portfolio_value)
        
        self.current_step += 1
        done = self.current_step >= self.n_steps - 1
        info = {"portfolio_value": self.portfolio_value, "pnl": pnl, "cost": cost}
        
        next_obs = self._get_observation() if not done else np.zeros_like(self._get_observation())
        return next_obs, reward, done, info

print("MarketTradingEnv scaffolding initialized successfully ✓")

## 4. Policy Network Architecture (Scaffolding)

In [ ]:
class ActorCriticTradingNetwork(nn.Module):
    """
    Actor-Critic architecture for reinforcement learning on market states.
    Input: Composite state vector of dim 34 (32-D market embedding + 2-D portfolio state).
    """
    def __init__(self, state_dim: int = 34, action_dim: int = 3, hidden_dim: int = 64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.actor = nn.Sequential(
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state: torch.Tensor):
        features = self.shared(state)
        action_probs = self.actor(features)
        state_value = self.critic(features)
        return action_probs, state_value

# Smoke test
policy = ActorCriticTradingNetwork()
dummy_obs = torch.randn(4, 34)
probs, val = policy(dummy_obs)
print(f"Policy Network Test — Obs: {tuple(dummy_obs.shape)} | Action Probs: {tuple(probs.shape)} | Value: {tuple(val.shape)} ✓")

## 5. Next Steps for Phase 6 (Reinforcement Learning)
1. **Agent Implementation:** Implement PPO (Proximal Policy Optimization) or DQN agent in `backend/agents/`.
2. **Environment Finalization:** Register modular trading environment in `backend/environments/`.
3. **Training & Validation:** Train agent on `train_embeddings.pt`, validate on `val_embeddings.pt`.
4. **Out-of-Sample Backtesting:** Backtest complete trading policy on `test_embeddings.pt`.
5. **Academic Benchmarks:** Compare RL agent against Buy-and-Hold, Equal-Weighted, and Momentum strategies.
6. **Reporting:** Save comprehensive reward curves, drawdowns, Sharpe ratios, and trade logs to `reports/reinforcement_learning/`.